# Proyecto: Análisis Big Data de Denuncias Policiales (2018-2026)
**Curso:** Big Data & Analytics  
**Autor:** Diego  
**Tecnología:** Apache Spark (PySpark) + Delta Lake

## 1. Introducción y Carga Incremental
En esta sección realizamos la extracción de datos desde el **Unity Catalog**. Aplicamos una lógica de carga incremental para integrar el nuevo volcado de datos (Diciembre 2025 - Enero 2026) con el repositorio histórico, garantizando la integridad mediante un **análisis de cardinalidad**.

In [0]:
import pyspark.sql.functions as F

# 1. Extracción (Extract): Leer Tablas Delta desde Unity Catalog
df_historico = spark.table("workspace.default.dataset_denuncias_policiales_ene_2018_a_nov_2025")
df_nuevo = spark.table("workspace.default.dataset_denuncias_policiales_ene_2018_a_ene_2026")

# 2. Transformación (Transform): Lógica Incremental
df_incremental = df_nuevo.filter(
    (F.col("ANIO") == 2026) | ((F.col("ANIO") == 2025) & (F.col("MES") == 12))
)

# 3. Carga en Memoria (Load): Unión de DataFrames
df_final = df_historico.unionByName(df_incremental)

# 4. Análisis de Cardinalidad
c_hist = df_historico.count()
c_inc = df_incremental.count()
c_fin = df_final.count()

print("--- REPORTE DE CARDINALIDAD ---")
print(f"Registros Históricos: {c_hist:,}")
print(f"Registros Incrementales: {c_inc:,}")
print(f"Total Registros Resultantes: {c_fin:,}")

if c_hist + c_inc == c_fin:
    print("\n VALIDACIÓN EXITOSA: La cardinalidad coincide. No hay duplicados ni pérdida de datos.")

## 2. Análisis Exploratorio de Datos (EDA)
A continuación, generamos visualizaciones interactivas para identificar patrones temporales y geográficos en la criminalidad a nivel nacional.

### Gráfico de Tendencias

In [0]:
# Evolución Anual de Denuncias
# Instrucción: Al ejecutar, haz clic en el icono "+" y selecciona "Line Chart"
'''
df_tendencia = df_final.groupBy("ANIO") \
                       .agg(F.sum("cantidad").alias("Total_Denuncias")) \
                       .orderBy("ANIO")
display(df_tendencia)'''
df_tendencia = df_final.filter(F.col("ANIO") < 2026) \
                       .groupBy("ANIO") \
                       .agg(F.sum("cantidad").alias("Total_Denuncias")) \
                       .orderBy("ANIO")

display(df_tendencia)

Databricks visualization. Run in Databricks to view.

###Gráfico de Zonas Rojas

In [0]:
# Top 10 Departamentos con mayor concentración de delitos
# Instrucción: Al ejecutar, selecciona "Bar Chart" y configura orientación Horizontal
df_zonas_rojas = df_final.groupBy("DPTO_HECHO_NEW") \
                         .agg(F.sum("cantidad").alias("Concentracion_Delitos")) \
                         .orderBy(F.col("Concentracion_Delitos").desc()) \
                         .limit(10)
display(df_zonas_rojas)

Databricks visualization. Run in Databricks to view.

###Mapa de Calor

In [0]:
# Análisis de Estacionalidad: Mes vs Año
# Instrucción: Al ejecutar, selecciona la visualización "Heatmap"
df_para_heatmap = df_final.groupBy("ANIO", "MES") \
                          .agg(F.sum("cantidad").alias("Total")) \
                          .orderBy("ANIO", "MES")

display(df_para_heatmap)

Databricks visualization. Run in Databricks to view.

###Proporción por Modalidad

In [0]:
# Análisis de Modalidades de Delito
df_modalidades = df_final.groupBy("P_MODALIDADES") \
                         .agg(F.sum("cantidad").alias("Total_Denuncias")) \
                         .orderBy(F.col("Total_Denuncias").desc())

display(df_modalidades)

Databricks visualization. Run in Databricks to view.